# Inference and Detection Pipeline
Real-time PPE detection using trained YOLOv8 model

## 1. Setup

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import os

model_path = '../models/best_model.pt'
conf_threshold = 0.5
iou_threshold = 0.45
test_image_dir = '../data/images/test'

print(f'Model path: {model_path}')
print(f'Model exists: {os.path.exists(model_path)}')
print(f'Test images dir: {test_image_dir}')
print(f'Test images exist: {os.path.exists(test_image_dir)}')

## 2. Load Model

In [ ]:
if os.path.exists(model_path):
    model = YOLO(model_path)
    print('Model loaded successfully')
    print(f'Classes: {model.names}')
else:
    print('Model not found. Please train the model first.')

## 3. Class Definitions

In [ ]:
class_names = {
    0: 'helmet',
    1: 'gloves',
    2: 'vest',
    3: 'boots',
    4: 'goggles',
    5: 'person'
}

mandatory_ppe = {
    'high_risk_area': ['helmet', 'vest', 'gloves', 'boots'],
    'standard_area': ['helmet', 'vest'],
    'office_area': ['helmet']
}

print('Class mapping defined')
print('Mandatory PPE rules defined')

## 4. Detection Helper Functions

In [ ]:
def run_inference(image_path, conf=0.5):
    try:
        results = model(image_path, conf=conf, iou=iou_threshold, max_det=300)
        return results
    except Exception as e:
        print(f'Error during inference: {e}')
        return None

def extract_detections(results):
    detections = {}
    
    for result in results:
        boxes = result.boxes.cpu().numpy()
        for box in boxes:
            class_id = int(box.cls[0])
            class_name = class_names.get(class_id, f'unknown_{class_id}')
            confidence = float(box.conf[0])
            
            x1, y1, x2, y2 = box.xyxy[0]
            
            if class_name not in detections:
                detections[class_name] = []
            
            detections[class_name].append({
                'confidence': confidence,
                'box': [float(x1), float(y1), float(x2), float(y2)],
                'center': [(float(x1)+float(x2))/2, (float(y1)+float(y2))/2]
            })
    
    return detections

print('Helper functions defined')

## 5. Test on Sample Image

In [ ]:
test_image_dir = '../data/construction-ppe/images/test'
image_files = [f for f in os.listdir(test_image_dir) 
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

if image_files:
    test_image = os.path.join(test_image_dir, image_files[0])
    print(f'Testing on: {test_image}')
    
    results = run_inference(test_image, conf=conf_threshold)
    
    if results:
        detections = extract_detections(results)
        print(f'\nDetections found:')
        for class_name, boxes in detections.items():
            print(f'  {class_name}: {len(boxes)} detections')
else:
    print('No test images found')

## 6. Batch Inference

In [ ]:
print(f'Running inference on {len(image_files)} test images...')
inference_results = {}

for img_file in image_files[:10]:
    img_path = os.path.join(test_image_dir, img_file)
    results = run_inference(img_path, conf=conf_threshold)
    
    if results:
        detections = extract_detections(results)
        inference_results[img_file] = detections

print(f'Inference completed on {len(inference_results)} images')

## 7. Results Summary

In [ ]:
print(f'\nInference Results Summary')
print('=' * 60)
total_detections = 0
class_summary = {}

for img_file, detections in inference_results.items():
    print(f'\n{img_file}:')
    for class_name, boxes in detections.items():
        print(f'  {class_name}: {len(boxes)} objects')
        if class_name not in class_summary:
            class_summary[class_name] = 0
        class_summary[class_name] += len(boxes)
    total_detections += sum(len(boxes) for boxes in detections.values())

print(f'\nTotal detections: {total_detections}')
print('\nClass distribution:')
for class_name, count in sorted(class_summary.items()):
    print(f'  {class_name}: {count}')